# 102 — Búsqueda léxica y BM25

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**BM25** (Robertson & Zaragoza, 2009) puntúa un documento frente a una consulta como
suma por término de tres factores:

```text
score(D,Q) = Σₜ IDF(t) · f(t,D)·(k₁+1) / ( f(t,D) + k₁·(1−b+b·|D|/avgdl) )
IDF(t) = ln(1 + (N − df + 0.5)/(df + 0.5))
```

- **IDF**: los términos raros discriminan; los omnipresentes casi no puntúan.
- **Saturación de TF** (`k₁`): repetir un término tiene rendimiento decreciente
  (techo en `k₁+1`) — cubrir más términos vale más que repetir uno.
- **Normalización por longitud** (`b`): penaliza documentos más largos que `avgdl`.

Fortaleza: coincidencia literal (IDs, siglas, nombres). Límite: *vocabulary mismatch*
("coche" no recupera "automóvil") — lo que resuelven los embeddings (clase 100).
El score BM25 ordena dentro de una consulta; **no** es comparable entre consultas.


## 🧮 Ejemplo de referencia

`Q = {gato, negro}`, `k₁=1.2`, `b=0.75`, `avgdl=6`, `IDF(gato)=IDF(negro)=ln(1.6)≈0.470`:

```text
D1 "el gato negro duerme" (4 tok):        0.544 + 0.544 = 1.088
D2 "el gato blanco y el gato gris…" (8):  0.591 + 0     = 0.591
D3 "el perro negro corre y ladra" (6):    0     + 0.470 = 0.470
Ranking: D1 > D2 > D3
```

D2 tiene "gato" dos veces y pierde: la saturación de TF y la penalización por longitud
hacen que cubrir ambos términos (D1) valga más que repetir uno.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("retrieval", seed=102)
show(result)


## Reflexión

1. En el ejemplo, D2 repite "gato" y aun así pierde contra D1. ¿Qué dos componentes de la fórmula producen ese resultado y qué pasaría con k₁ = 100?
2. ¿Por qué los scores BM25 de dos consultas distintas no son comparables entre sí, y qué problema causa eso al fusionar rankings (anticipa la clase 103)?
3. Da dos ejemplos de consultas de tu dominio donde BM25 debería ganar a los embeddings, y dos donde debería perder. ¿Cómo lo comprobarías?
